# Stage 6 — Experiment 0: Recluster Saved Embeddings

**Goal:** zero-training re-evaluation of all 234 saved fused embeddings from Stage 5.

**Why:** before training anything new, check if the embeddings we already have are
better than we thought — just clustered or normalized differently. 4 transforms
(raw / standard / l2 / pca24) x 3 heads (kmeans-50 / spherical / gmm) = 12 evals
per embedding. Unsupervised metrics for selection; post-hoc ARI quarantined to
reporting only (labels never touch training — there is no training).

**Output:** `runs/s6-exp0/` — per-embedding results + summary ranking.


In [ ]:
import os, sys, json, subprocess, random, glob, time
import numpy as np
import pandas as pd

for pkg in ["scikit-learn", "pandas", "scipy"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import normalize

def set_seed(s):
    random.seed(s); np.random.seed(s)

print("imports ok")


In [ ]:
# ---------- GitHub push (same pattern as stage5) ----------
REPO = "/kaggle/working/my_research"
REPO_URL = "https://github.com/rifatahsanpul0k/my_research.git"
def sh(*args):
    subprocess.run(list(args), check=True, cwd=REPO)
os.makedirs(REPO, exist_ok=True)
OUT = os.path.join(REPO, "runs", "s6-exp0")
os.makedirs(OUT, exist_ok=True)
try:
    from kaggle_secrets import UserSecretsClient
    import base64, shutil
    _sec = UserSecretsClient()
    _b64 = base64.b64encode(f"x-access-token:{_sec.get_secret('GITHUB_TOKEN')}".encode()).decode()
    _GIT = ["git", "-c", f"http.extraHeader=Authorization: Basic {_b64}"]
    if not os.path.isdir(os.path.join(REPO, ".git")):
        if os.listdir(REPO):
            bak = REPO + "_stalebak"
            if os.path.exists(bak):
                shutil.rmtree(bak)
            shutil.move(REPO, bak)
            os.makedirs(REPO, exist_ok=True)
        subprocess.run(_GIT + ["clone", REPO_URL, REPO], check=True)
        sh("git", "config", "user.name", "kaggle-runner")
        sh("git", "config", "user.email", "kaggle-runner@local")
    else:
        sh(*_GIT, "pull", "--rebase", "origin", "main")
    GIT_OK = True
    print("GitHub sync ready")
except Exception as e:
    GIT_OK = False
    _safe = str(e).replace(_b64, "***") if "_b64" in dir() else str(e)
    print("GitHub sync unavailable (local save only):", _safe)

def _scrub(s):
    try:
        return str(s).replace(_b64, "***")
    except NameError:
        return str(s)

def git_push(paths, msg):
    if not GIT_OK:
        return
    try:        # commit FIRST, then pull --rebase, then push
        sh("git", "add", *paths)
        r = subprocess.run(["git", "status", "--porcelain"], capture_output=True,
                           text=True, cwd=REPO)
        if r.stdout.strip():
            sh("git", "commit", "-m", msg)
        sh(*_GIT, "pull", "--rebase", "origin", "main")
        sh(*_GIT, "push", "origin", "main")
        print("pushed:", msg)
    except Exception as e:
        print("push failed (continuing):", _scrub(e))


In [ ]:
# ---------- data root (same MARKER pattern as stage5) ----------
MARKER = "10x_human_lymph_node_A1"
def find_data_root():
    for base in ["/kaggle/input/spatial-multiomics-6datasets-private",
                 "/kaggle/working/spatial-multiomics-6datasets-private",
                 "/kaggle/working"]:
        if os.path.isdir(os.path.join(base, MARKER)):
            return base
    return None

DATA_ROOT = find_data_root()
if DATA_ROOT is None:
    # download fallback (same as stage5)
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    kd = os.path.expanduser("~/.kaggle"); os.makedirs(kd, exist_ok=True)
    json.dump({"username": _s.get_secret("KAGGLE_USERNAME"), "key": _s.get_secret("KAGGLE_KEY")},
              open(os.path.join(kd, "kaggle.json"), "w"))
    os.chmod(os.path.join(kd, "kaggle.json"), 0o600)
    subprocess.run(["kaggle", "datasets", "download", "-d", "pulokpulok/spatial-multiomics-6datasets-private",
                    "-p", "/kaggle/working", "--unzip"], check=True)
    DATA_ROOT = find_data_root()
if DATA_ROOT is None:
    raise FileNotFoundError("dataset folders not found under /kaggle/working: "
                            + str(sorted(os.listdir("/kaggle/working"))))
print("DATA_ROOT:", DATA_ROOT)

# SAME spec as stage5 — labels live in per-dataset folders, never in the repo.
GT_SPEC = {
    "A1":  dict(folder="10x_human_lymph_node_A1", gt_file="annotation.csv", gt_col="manual-anno"),
    "D1":  dict(folder="10x_human_lymph_node_D1", gt_file="annotation.csv", gt_col="manual-anno"),
    "E11": dict(folder="Mouse_Brain_E11_S1", gt_file="anno.csv", gt_col="cluster"),
    "E13": dict(folder="Mouse_Brain_E13_S1", gt_file="anno.csv", gt_col="cluster"),
    "E15": dict(folder="Mouse_Brain_E15_S1", gt_file="anno.csv", gt_col="cluster"),
    "E18": dict(folder="Mouse_Brain_E18_S1", gt_file="anno.csv", gt_col="cluster"),
}
GT = {}
for ds, spec in GT_SPEC.items():
    p = os.path.join(DATA_ROOT, spec["folder"], spec["gt_file"])
    assert os.path.exists(p), f"missing {p}"
    df = pd.read_csv(p, index_col=0)
    GT[ds] = df[spec["gt_col"]].astype(str).values
    print(ds, "gt:", GT[ds].shape, "classes:", len(np.unique(GT[ds])))

# declared k (same exception as stage5: applied identically, disclosed)
DECLARED_K = {"A1": 10, "D1": 11, "E11": 8, "E13": 12, "E15": 12, "E18": 14}


In [ ]:
# ---------- transforms x heads ----------
def apply_transform(Z, name):
    if name == "raw":
        return Z
    if name == "standard":
        return StandardScaler().fit_transform(Z)
    if name == "l2":
        return normalize(Z, norm="l2")
    if name == "pca24":
        return PCA(n_components=min(24, Z.shape[1]), random_state=42).fit_transform(Z)
    raise ValueError(name)

def cluster(Z, head, k, seed=42):
    if head == "kmeans":
        return KMeans(n_clusters=k, n_init=50, random_state=seed).fit_predict(Z)
    if head == "spherical":
        Zn = normalize(Z, norm="l2")
        return KMeans(n_clusters=k, n_init=50, random_state=seed).fit_predict(Zn)
    if head == "gmm":
        return GaussianMixture(n_components=k, n_init=5, random_state=seed).fit_predict(Z)
    raise ValueError(head)

TRANSFORMS = ["raw", "standard", "l2", "pca24"]
HEADS = ["kmeans", "spherical", "gmm"]

# ---------- collect embeddings ----------
S5 = os.path.join(REPO, "runs", "s5-notebook")
emb_files = []  # (combo, ds, seed, path)
for combo in sorted(os.listdir(S5)):
    cd = os.path.join(S5, combo)
    if not os.path.isdir(cd):
        continue
    for ds in sorted(os.listdir(cd)):
        dd = os.path.join(cd, ds)
        if not os.path.isdir(dd):
            continue
        for seed_d in sorted(os.listdir(dd)):
            p = os.path.join(dd, seed_d, "fused_embedding.npy")
            if os.path.exists(p):
                emb_files.append((combo, ds, seed_d, p))
print(f"found {len(emb_files)} embeddings")

# ---------- evaluate ----------
results = []
for combo, ds, seed_d, p in emb_files:
    Z = np.load(p)
    k = DECLARED_K[ds]
    gt = GT[ds]
    assert Z.shape[0] == len(gt), f"n mismatch {p}: {Z.shape[0]} vs {len(gt)}"
    for t in TRANSFORMS:
        Zt = apply_transform(Z, t)
        for h in HEADS:
            labs = cluster(Zt, h, k)
            sil = float(silhouette_score(Zt, labs)) if len(np.unique(labs)) > 1 else -1.0
            ari = float(adjusted_rand_score(gt, labs))
            results.append(dict(combo=combo, dataset=ds, seed=seed_d,
                                transform=t, head=h, silhouette=sil, ari=ari,
                                n_clusters=int(len(np.unique(labs)))))
    if len(results) % 500 == 0:
        print(f"  {len(results)} evals done...")

print(f"total {len(results)} evals")
json.dump(results, open(os.path.join(OUT, "exp0_results.json"), "w"), indent=1)
git_push(["runs/s6-exp0/exp0_results.json"], "s6-exp0: raw results")


In [ ]:
# ---------- summary: best transform+head per combo per dataset ----------
from collections import defaultdict
import numpy as np

by_combo_ds = defaultdict(list)
for r in results:
    by_combo_ds[(r["combo"], r["dataset"])].append(r)

print("\n=== best (transform, head) per combo per dataset (by mean ARI over seeds) ===\n")
summary = []
for (combo, ds) in sorted(by_combo_ds):
    rows = by_combo_ds[(combo, ds)]
    # group by (transform, head), average over seeds
    grp = defaultdict(list)
    for r in rows:
        grp[(r["transform"], r["head"])].append(r["ari"])
    best_th, best_ari = max(grp.items(), key=lambda kv: np.mean(kv[1]))
    t, h = best_th
    sils = [r["silhouette"] for r in rows if (r["transform"], r["head"]) == best_th]
    summary.append(dict(combo=combo, dataset=ds, transform=t, head=h,
                        mean_ari=float(np.mean(best_ari)),
                        mean_sil=float(np.mean(sils))))
    print(f"{combo:24} {ds:4} {t:9} {h:9} ARI={np.mean(best_ari):.3f} sil={np.mean(sils):.3f}")

print("\n=== cross-dataset mean ARI per combo (best transform+head each) ===\n")
combo_aris = defaultdict(list)
for s in summary:
    combo_aris[s["combo"]].append(s["mean_ari"])
for combo in sorted(combo_aris, key=lambda c: -np.mean(combo_aris[c])):
    aris = combo_aris[combo]
    print(f"{combo:24} mean ARI={np.mean(aris):.3f}  " +
          " ".join(f"{a:.3f}" for a in aris))

json.dump(summary, open(os.path.join(OUT, "exp0_summary.json"), "w"), indent=1)
git_push(["runs/s6-exp0/exp0_summary.json"], "s6-exp0: summary")
print("\nAll Exp0 results saved under", OUT)


## Notes
- **Firewall:** labels are read once for quarantined post-hoc ARI only. No training happens here — pure re-evaluation.
- **Declared k** is the same exception as stage5 (ground-truth label counts, applied identically to every config, disclosed).
- **Resume:** re-running overwrites `runs/s6-exp0/` — results are deterministic given the saved embeddings.
- **Manual run:** upload to Kaggle → attach `pulokpulok/spatial-multiomics-6datasets-private` as input → add the `GITHUB_TOKEN` secret for auto-push → CPU is fine → Save & Run All.
